# Parameter-Matched MLP Baseline — MNIST

In [ ]:
import time
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# =========================
# 1. Parameter-matched MLP baseline
#    hidden_dim=216 -> ~219,466 params, matching RecKAN's ~219,919 params
#    (RecKAN: 784->64->64->10 with degree=3, i.e. 4 coeffs per edge)
# =========================

class SimpleMLP_MNIST(nn.Module):
    def __init__(self, input_dim=784, hidden_dim=216, num_classes=10):
        super().__init__()
        self.flatten = nn.Flatten()

        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.norm1 = nn.LayerNorm(hidden_dim)
        self.act1 = nn.ReLU()
        self.drop1 = nn.Dropout(0.1)

        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.norm2 = nn.LayerNorm(hidden_dim)
        self.act2 = nn.ReLU()
        self.drop2 = nn.Dropout(0.1)

        self.fc3 = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.norm1(x)
        x = self.act1(x)
        x = self.drop1(x)

        x = self.fc2(x)
        x = self.norm2(x)
        x = self.act2(x)
        x = self.drop2(x)

        x = self.fc3(x)
        return x


# =========================
# 2. Data loaders
# =========================

MNIST_MEAN = (0.1307,)
MNIST_STD = (0.3081,)


def make_mnist_loaders(batch_size=64):
    train_tf = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(MNIST_MEAN, MNIST_STD)
    ])
    test_tf = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(MNIST_MEAN, MNIST_STD)
    ])

    train_ds = datasets.MNIST('./data_mnist',
                              train=True,
                              download=True,
                              transform=train_tf)
    test_ds = datasets.MNIST('./data_mnist',
                             train=False,
                             download=True,
                             transform=test_tf)

    train_loader = DataLoader(train_ds, batch_size=batch_size,
                              shuffle=True, num_workers=0)
    test_loader = DataLoader(test_ds, batch_size=1000,
                             shuffle=False, num_workers=0)
    return train_loader, test_loader


# =========================
# 3. Training loop (identical structure/logging to the RecursivePolyKAN script)
# =========================

def train_mnist(epochs=50, batch_size=64, device='cuda', hidden_dim=216):
    print("=" * 60)
    print(f"Parameter-matched MLP (784->{hidden_dim}->{hidden_dim}->10) on MNIST")
    print(f"Device: {device} | Epochs: {epochs} | Batch: {batch_size}")
    print("=" * 60)

    train_loader, test_loader = make_mnist_loaders(batch_size=batch_size)

    model = SimpleMLP_MNIST(
        input_dim=784, hidden_dim=hidden_dim, num_classes=10
    ).to(device)

    n_params = sum(p.numel() for p in model.parameters())
    print(f"Total trainable parameters: {n_params:,}  "
          f"(RecKAN with degree=3, hidden=64 has ~219,919)")

    optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    criterion = nn.CrossEntropyLoss()

    best_test_acc = 0.0

    for epoch in range(1, epochs + 1):
        model.train()
        start = time.time()
        epoch_loss = 0.0

        for data, target in train_loader:
            data, target = data.to(device), target.to(device)
            optimizer.zero_grad()
            out = model(data)
            loss = criterion(out, target)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            epoch_loss += loss.item()

        # eval روی test
        model.eval()
        correct_te, total_te = 0, 0
        with torch.no_grad():
            for data, target in test_loader:
                data, target = data.to(device), target.to(device)
                pred = model(data).argmax(dim=1)
                correct_te += pred.eq(target).sum().item()
                total_te += target.size(0)
        test_acc = 100.0 * correct_te / total_te
        best_test_acc = max(best_test_acc, test_acc)

        avg_loss = epoch_loss / len(train_loader)
        elapsed = time.time() - start

        print(f"Epoch {epoch:2d}/{epochs} | Loss: {avg_loss:.4f} | "
              f"Test Acc: {test_acc:.2f}% | Best Test: {best_test_acc:.2f}% "
              f"| Time: {elapsed:.1f}s")

        scheduler.step()

    return model


# =========================
# 4. Main
# =========================

if __name__ == '__main__':
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    EPOCHS = 20
    BATCH_SIZE = 64
    HIDDEN_DIM = 216   # param-matched to RecKAN (~220k params)

    model = train_mnist(epochs=EPOCHS,
                        batch_size=BATCH_SIZE,
                        device=DEVICE,
                        hidden_dim=HIDDEN_DIM)

    torch.save(model.state_dict(), 'simple_mlp_mnist_paramMatched.pth')
    print("Model saved: simple_mlp_mnist_paramMatched.pth")